# Campaign Effectiveness Analysis - Dunnhumby Complete Journey

**Question**: Did households exposed to a major loyalty campaign show a 
statistically significant increase in spend during the campaign period, 
compared to an equivalent period before it and was this specific to the 
campaign, or part of a broader spending trend?

In [4]:
!pip install duckdb


In [5]:
import pandas as pd
import duckdb
import os
print(os.listdir())

['hh_demographic.csv', 'causal_data.csv', 'campaign_analysis.ipynb', 'coupon.csv', 'Untitled.ipynb', 'campaign_table.csv', '.ipynb_checkpoints', 'coupon_redempt.csv', 'product.csv', 'campaign_desc.csv', 'transaction_data.csv']


In [6]:
campaign_desc = pd.read_csv('campaign_desc.csv')
campaign_table = pd.read_csv('campaign_table.csv')

print(campaign_desc.head())
print(campaign_desc.shape)

  DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
0       TypeB        24        659      719
1       TypeC        15        547      708
2       TypeB        25        659      691
3       TypeC        20        615      685
4       TypeB        23        646      684
(30, 4)


## Selecting a campaign
Campaigns vary widely in size. I selected the largest campaign by household 
count to maximise statistical power (Campaign 18).

In [18]:
query1 = """
SELECT 
    CAMPAIGN,
    COUNT(DISTINCT household_key) AS num_households
FROM 'campaign_table.csv'
GROUP BY CAMPAIGN
ORDER BY num_households DESC
"""
result1 = duckdb.sql(query1).df()
print(result1.head(10))

   CAMPAIGN  num_households
0        18            1133
1        13            1077
2         8            1076
3        30             361
4        26             332
5        22             276
6        20             244
7        14             224
8        11             214
9        17             202


## Defining before/during windows
Campaign 18 ran for 55 days (day 587–642). I used an equal-length baseline 
window immediately prior (day 532–587) for a fair before/after comparison.

In [19]:
query2 = """
SELECT START_DAY, END_DAY
FROM 'campaign_desc.csv'
WHERE CAMPAIGN = 18
"""
result2 = duckdb.sql(query2).df()
print(result2)


   START_DAY  END_DAY
0        587      642


In [21]:
query3 = """
SELECT household_key
FROM 'campaign_table.csv'
WHERE CAMPAIGN = 18
"""
result3 = duckdb.sql(query3).df()
print(result3.shape)
print(result3.head())

(1133, 1)
   household_key
0              1
1              6
2              7
3              8
4             14


In [20]:
transactions = duckdb.sql("SELECT * FROM 'transaction_data.csv' LIMIT 5").df()
print(transactions)

   household_key    BASKET_ID  DAY  PRODUCT_ID  QUANTITY  SALES_VALUE  \
0           2375  26984851472    1     1004906         1         1.39   
1           2375  26984851472    1     1033142         1         0.82   
2           2375  26984851472    1     1036325         1         0.99   
3           2375  26984851472    1     1082185         1         1.21   
4           2375  26984851472    1     8160430         1         1.50   

   STORE_ID  RETAIL_DISC TRANS_TIME  WEEK_NO  COUPON_DISC  COUPON_MATCH_DISC  
0       364        -0.60       1631        1          0.0                0.0  
1       364         0.00       1631        1          0.0                0.0  
2       364        -0.30       1631        1          0.0                0.0  
3       364         0.00       1631        1          0.0                0.0  
4       364        -0.39       1631        1          0.0                0.0  


In [10]:
query = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend
FROM 'transaction_data.csv'
WHERE household_key IN (
    SELECT household_key FROM 'campaign_table.csv' WHERE CAMPAIGN = 18
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""

result = duckdb.sql(query).df()
print(result.head(10))
print(result.shape)

   household_key  period  total_spend
0            510  before       766.92
1           1791  before       327.44
2            853  before      1137.41
3           1795  before      1010.77
4           2453  before       242.37
5            656  before       843.05
6            534  before       110.99
7            820  before       443.36
8           1448  before         8.49
9            121  before       137.85
(2219, 3)


## Reshaping and handling missing periods
Households with no transactions in a given period are treated as £0 spend 
(not dropped), since they may represent customers activated by the campaign.

In [11]:
pivot = result.pivot(index='household_key', columns='period', values='total_spend')
pivot = pivot.fillna(0)  # households with no purchases in a period = £0 spend, not missing
print(pivot.head(10))
print(pivot.shape)

period          before   during
household_key                  
1               342.71   465.67
2               236.35   201.86
6               465.11   406.07
7               392.31   299.41
8               464.81   478.20
13             1092.75  1538.71
14              148.96   132.54
17              154.72   485.42
18              583.89   872.78
19             1424.94  1328.67
(1123, 2)


In [12]:
pivot['change'] = pivot['during'] - pivot['before']
print(f"Average spend before: £{pivot['before'].mean():.2f}")
print(f"Average spend during: £{pivot['during'].mean():.2f}")
print(f"Average change: £{pivot['change'].mean():.2f}")

Average spend before: £430.86
Average spend during: £465.63
Average change: £34.77


## Testing significance
Used a paired t-test since we're comparing each household against itself 
across two time periods.

In [13]:
from scipy import stats

t_stat, p_value = stats.ttest_rel(pivot['during'], pivot['before'])
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.5f}")

t-statistic: 5.423
p-value: 0.00000


## Why a control group is needed

A before/after comparison alone cannot tell us whether the increase in spend 
was caused by the campaign, or reflects a broader trend affecting all 
customers during this period (e.g. seasonality). To test this, I compared 
the same before/during change for a control group of households not exposed 
to any campaign running during this window.

## Identifying overlapping campaigns

Several other campaigns overlapped the analysis window. To build a clean 
control group, I excluded households in any of these campaigns, not just 
Campaign 18.

In [14]:
overlapping = duckdb.sql("""
    SELECT CAMPAIGN, START_DAY, END_DAY
    FROM 'campaign_desc.csv'
    WHERE START_DAY < 642 AND END_DAY > 532
""").df()
print(overlapping)

   CAMPAIGN  START_DAY  END_DAY
0        15        547      708
1        20        615      685
2        21        624      656
3        22        624      656
4        18        587      642
5        19        603      635
6        17        575      607
7        14        531      596
8        16        561      593
9        13        504      551


In [15]:
query_control = """
SELECT
    household_key,
    CASE 
        WHEN DAY >= 532 AND DAY < 587 THEN 'before'
        WHEN DAY >= 587 AND DAY < 642 THEN 'during'
    END AS period,
    SUM(SALES_VALUE) AS total_spend
FROM 'transaction_data.csv'
WHERE household_key NOT IN (
    SELECT household_key FROM 'campaign_table.csv' 
    WHERE CAMPAIGN IN (13,14,15,16,17,18,19,20,21,22)
)
AND DAY >= 532 AND DAY < 642
GROUP BY household_key, period
"""

control_result = duckdb.sql(query_control).df()
print(control_result.shape)

(1744, 3)


## Calculating control group spend


In [16]:
control_pivot = control_result.pivot(index='household_key', columns='period', values='total_spend')
control_pivot = control_pivot.fillna(0)
control_pivot['change'] = control_pivot['during'] - control_pivot['before']

print(f"Control households: {control_pivot.shape[0]}")
print(f"Average spend before: £{control_pivot['before'].mean():.2f}")
print(f"Average spend during: £{control_pivot['during'].mean():.2f}")
print(f"Average change: £{control_pivot['change'].mean():.2f}")

t_stat_control, p_value_control = stats.ttest_rel(control_pivot['during'], control_pivot['before'])
print(f"t-statistic: {t_stat_control:.3f}")
print(f"p-value: {p_value_control:.5f}")

Control households: 978
Average spend before: £119.26
Average spend during: £140.92
Average change: £21.66
t-statistic: 5.459
p-value: 0.00000


**Findings**:
- Campaign 18 households: average spend rose from £430.86 to £465.63 
  (+£34.77, +8.1%), statistically significant (p < 0.001)
- Control households: average spend rose from £119.26 to £140.92 
  (+£21.66, +18.2%), also statistically significant (p < 0.001)

**Interpretation**: Both groups increased spend over this period, suggesting 
some of the rise reflects a general trend (e.g. seasonality) rather than the 
campaign alone. Campaign households increased more in absolute pounds, but 
less in percentage terms. Campaign households also had 3.6x higher baseline 
spend than the control group before the campaign even started, indicating 
Campaign 18 was not targeted at a random sample of customers, but likely at 
already high-value shoppers — a clear instance of selection bias.

**Limitation**: This analysis cannot cleanly separate the campaign's effect 
from broader spending trends or pre-existing differences between the two 
groups. A stronger analysis would match campaign and control households on 
similar baseline spend before comparing their changes, to better isolate the 
campaign's true incremental effect.

**Conclusion**: The data does not provide clear evidence that Campaign 18 
caused incremental spend beyond what would have happened anyway. The result 
is genuinely ambiguous — campaign households show a larger absolute increase, 
but a smaller relative increase, and started from a very different baseline. 
Further work using baseline-matched control groups would be needed to draw a 
confident conclusion about the campaign's true effect.